# 附錄 10.1：串連提示詞（Chaining Prompts）

> 📚 **第 2 單元：提示詞工程互動教學**｜附錄 1 / 共 3 篇（超越標準提示詞）
> ⏱️ 預估時間：30 分鐘

- [課程](#課程)
- [範例遊樂場](#範例遊樂場)

## 🎯 這一章你會學到

- 什麼是「**串連提示詞（prompt chaining）**」：把 Claude 上一次的回答，放進下一次的提示詞裡
- 請 Claude **檢查並修正**自己的答案
- 請 Claude **改進**自己的作品
- 把一次呼叫的結果，當作另一次呼叫的輸入

## 準備工作

執行下面這個準備格，載入你的 API 金鑰，並建立 `get_completion` 輔助函式。

In [ ]:
%pip install "anthropic<1"

# Import python's built-in regular expression library
import re
import anthropic

# Retrieve the API_KEY & MODEL_NAME variables from the IPython store
%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

# Has been rewritten to take in a messages list of arbitrary length
def get_completion(messages, system_prompt=""):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        system=system_prompt,
        messages=messages
    )
    return message.content[0].text

**注意：這一版的 `get_completion` 不一樣了！** 程式碼的註解說：「已改寫成可以接收任意長度的訊息串列」。
- 以前是 `get_completion(prompt)`，傳入**一段文字**，函式自己幫你包成一則 user 訊息。
- 現在是 `get_completion(messages)`，你要自己傳入**整個訊息串列**（[0-4 第 7 節](../00_before_you_start/04_python_crash_course.md#7-把串列和字典組合起來claude-的訊息格式)），這樣才能放入多回合的對話。

---

## 課程

俗話說：「好文章是改出來的。」事實證明，**只要請 Claude 改進，它常常就能提升回答的準確度**！

請 Claude「再想一次」的方法有很多種。那些請人類「再檢查一次自己的工作」時感覺很自然的說法，通常對 Claude 也同樣有效。（想看更多「何時、如何使用串連提示詞」的例子，請參考我們的[串連提示詞文件](https://docs.anthropic.com/claude/docs/chain-prompts)。）

> 🧩 **白話小補充：什麼是「串連」？** 就像接力賽：第一棒（第一次呼叫 Claude）跑完，把結果交給第二棒（第二次呼叫），第二棒根據第一棒的結果繼續做事。程式碼裡的做法，就是把 Claude 的第一次回答放進 `messages` 裡的 `assistant` 訊息，再加一則新的 `user` 訊息提出下一個要求。

### 範例

在這個例子中，我們請 Claude 想出十個字⋯⋯但其中有一個或多個不是真正的英文單字。

In [ ]:
# Initial prompt
first_user = "Name ten words that all end with the exact letters 'ab'."

# API messages array
messages = [
    {
        "role": "user",
        "content": first_user
    }
]

# Store and print Claude's response
first_response = get_completion(messages)
print(first_response)

（「說出十個都以 'ab' 這兩個字母結尾的單字。」——Claude 有時候會編出不存在的字。）

**請 Claude 讓它的答案更準確**，就能修正錯誤！

下面，我們把 Claude 上面那個錯誤的回答拿下來，並在對話中再加一個回合，請 Claude 修正它之前的答案。

In [ ]:
second_user = "Please find replacements for all 'words' that are not real words."

# API messages array
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Print Claude's response
print("------------------------ Full messsages array with variable substutions ------------------------")
print(messages)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(messages))

**逐行解說：**
- `second_user`：第二個問題——「請把所有不是真正單字的『字』換掉。」
- `messages` 裡有三則訊息：
  1. user：第一個問題
  2. assistant：`first_response`（Claude 第一次的回答，從上一格拿來的）
  3. user：第二個問題
- `print(messages)` 先印出完整的訊息串列，讓你看清楚送了什麼。

但是，Claude 修改答案，是不是只因為我們叫它改？如果一開始的答案就已經是正確的呢？Claude 會失去自信嗎？下面我們把 `first_response` 換成一個**正確的**回答，再請它檢查一次。

In [ ]:
first_user = "Name ten words that all end with the exact letters 'ab'."

first_response = """Here are 10 words that end with the letters 'ab':

1. Cab
2. Dab
3. Grab
4. Gab
5. Jab
6. Lab
7. Nab
8. Slab
9. Tab
10. Blab"""

second_user = "Please find replacements for all 'words' that are not real words."

# API messages array
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Print Claude's response
print("------------------------ Full messsages array with variable substutions ------------------------")
print(messages)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(messages))

（這次的 `first_response` 是我們自己寫好的、完全正確的 10 個單字：Cab、Dab、Grab、Gab、Jab、Lab、Nab、Slab、Tab、Blab。）

你可能會注意到：如果你把上面這格執行好幾次，Claude 大部分時候會保留原本的單字，但偶爾還是會把單字改掉——即使它們全部都已經是正確的了。我們能怎麼減少這種情況呢？根據第 8 章，我們可以**給 Claude 一條退路**！我們再試一次。

In [ ]:
first_user = "Name ten words that all end with the exact letters 'ab'."

first_response = """Here are 10 words that end with the letters 'ab':

1. Cab
2. Dab
3. Grab
4. Gab
5. Jab
6. Lab
7. Nab
8. Slab
9. Tab
10. Blab"""

second_user = "Please find replacements for all 'words' that are not real words. If all the words are real words, return the original list."

# API messages array
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Print Claude's response
print("------------------------ Full messsages array with variable substutions ------------------------")
print(messages)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(messages))

（第二個問題加了一句：「**如果所有的字都是真正的單字，就回傳原本的清單。**」）

多執行上面的程式碼幾次，你會發現 Claude 現在更能「堅持立場」了。

你也可以用串連提示詞**請 Claude 把回答變得更好**。下面我們請 Claude 先寫一個故事，然後再改進它寫的故事。每個人的喜好不同，但很多人可能會同意 Claude 的第二個版本比較好。

首先，讓 Claude 產生故事的第一個版本。

In [ ]:
# Initial prompt
first_user = "Write a three-sentence short story about a girl who likes to run."

# API messages array
messages = [
    {
        "role": "user",
        "content": first_user
    }
]

# Store and print Claude's response
first_response = get_completion(messages)
print(first_response)

（「寫一個關於一個喜歡跑步的女孩、只有三句話的短篇故事。」）

接著，讓 Claude 改進它的初稿。

In [ ]:
second_user = "Make the story better."

# API messages array
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Print Claude's response
print("------------------------ Full messsages array with variable substutions ------------------------")
print(messages)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(messages))

（第二個要求只有一句：「把故事寫得更好。」）

這種「替換」的方式非常強大。我們一直在用替換用的佔位符來傳入清單、單字、Claude 之前的回答等等。你也可以**用替換來做我們稱為「函式呼叫（function calling）」的事：請 Claude 執行某個函式，然後把那個函式的結果拿回來，再請 Claude 根據這些結果做更多事**。它的運作方式和其他替換一樣。下一篇附錄會詳細介紹。

下面再舉一個例子：把一次呼叫 Claude 的結果，放進另一個更長的呼叫裡。我們先從第一個提示詞開始（這次還包含了預先填寫 Claude 的回答）。

In [ ]:
first_user = """Find all names from the below text:

"Hey, Jesse. It's me, Erin. I'm calling about the party that Joey is throwing tomorrow. Keisha said she would come and I think Mel will be there too."""

prefill = "<names>"

# API messages array
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": prefill
    
    }
]

# Store and print Claude's response
first_response = get_completion(messages)
print("------------------------ Full messsages array with variable substutions ------------------------")
print(messages)
print("\n------------------------------------- Claude's response -------------------------------------")
print(first_response)

**提示詞翻譯**：「找出下面這段文字裡所有的人名：『嘿，Jesse。是我，Erin。我打來是要說 Joey 明天要辦的派對。Keisha 說她會來，我想 Mel 也會到。』」prefill 是 `<names>`。

我們把這份人名清單傳進另一個提示詞裡。

In [ ]:
second_user = "Alphabetize the list."

# API messages array
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": prefill + "\n" + first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Print Claude's response
print("------------------------ Full messsages array with variable substutions ------------------------")
print(messages)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(messages))

**逐行解說：**
- `second_user = "Alphabetize the list."`：「把清單按照字母順序排列。」
- `"content": prefill + "\n" + first_response`：assistant 訊息的內容 = 我們 prefill 的 `<names>` + 換行 + Claude 接著寫的內容。因為 Claude 回傳的內容**不包含** prefill 的部分，所以要自己把它接回去，對話紀錄才完整。

學完串連提示詞之後，請前往附錄 10.2，學習怎麼用串連提示詞實作函式呼叫。

---

## 範例遊樂場

這是一個讓你**自由實驗**的區域，你可以試玩這一課中出現的提示詞範例，調整提示詞，看看會怎麼影響 Claude 的回答。

In [ ]:
# Initial prompt
first_user = "Name ten words that all end with the exact letters 'ab'."

# API messages array
messages = [
    {
        "role": "user",
        "content": first_user
    }
]

# Store and print Claude's response
first_response = get_completion(messages)
print(first_response)

In [ ]:
second_user = "Please find replacements for all 'words' that are not real words."

# API messages array
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Print Claude's response
print("------------------------ Full messsages array with variable substutions ------------------------")
print(messages)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(messages))

In [ ]:
first_user = "Name ten words that all end with the exact letters 'ab'."

first_response = """Here are 10 words that end with the letters 'ab':

1. Cab
2. Dab
3. Grab
4. Gab
5. Jab
6. Lab
7. Nab
8. Slab
9. Tab
10. Blab"""

second_user = "Please find replacements for all 'words' that are not real words."

# API messages array
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Print Claude's response
print("------------------------ Full messsages array with variable substutions ------------------------")
print(messages)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(messages))

In [ ]:
first_user = "Name ten words that all end with the exact letters 'ab'."

first_response = """Here are 10 words that end with the letters 'ab':

1. Cab
2. Dab
3. Grab
4. Gab
5. Jab
6. Lab
7. Nab
8. Slab
9. Tab
10. Blab"""

second_user = "Please find replacements for all 'words' that are not real words. If all the words are real words, return the original list."

# API messages array
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Print Claude's response
print("------------------------ Full messsages array with variable substutions ------------------------")
print(messages)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(messages))

In [ ]:
# Initial prompt
first_user = "Write a three-sentence short story about a girl who likes to run."

# API messages array
messages = [
    {
        "role": "user",
        "content": first_user
    }
]

# Store and print Claude's response
first_response = get_completion(messages)
print(first_response)

In [ ]:
second_user = "Make the story better."

# API messages array
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Print Claude's response
print("------------------------ Full messsages array with variable substutions ------------------------")
print(messages)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(messages))

In [ ]:
first_user = """Find all names from the below text:

"Hey, Jesse. It's me, Erin. I'm calling about the party that Joey is throwing tomorrow. Keisha said she would come and I think Mel will be there too."""

prefill = "<names>"

# API messages array
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": prefill
    
    }
]

# Store and print Claude's response
first_response = get_completion(messages)
print("------------------------ Full messsages array with variable substutions ------------------------")
print(messages)
print("\n------------------------------------- Claude's response -------------------------------------")
print(first_response)

In [ ]:
second_user = "Alphabetize the list."

# API messages array
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": prefill + "\n" + first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Print Claude's response
print("------------------------ Full messsages array with variable substutions ------------------------")
print(messages)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(messages))

---

## 📌 重點整理

- **串連提示詞** = 把 Claude 上一次的回答放進對話紀錄（`assistant` 訊息），再提出下一個要求。
- 用途：請 Claude **檢查並修正**（「把不是真正單字的換掉」）、**改進作品**（「把故事寫得更好」）、**把結果交給下一步處理**（先找人名 → 再排序）。
- 請 Claude 檢查時，記得**給退路**（「如果都正確，就回傳原本的清單」），避免它為了改而改。
- 使用 prefill 時，要把 prefill 的文字**接回**回答前面，對話紀錄才完整。

👉 下一篇：[附錄 10.2：工具使用](10_2_Appendix_Tool_Use.md)